In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

C:\Users\Ritesh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
import tensorflow as tf
import keras

In [3]:
from keras.models import Sequential
from keras.layers import Conv2D , MaxPool2D , Flatten , Dense , Dropout
from tensorflow.keras.utils import to_categorical

In [4]:
import os
import cv2  #computer vision cv2images would be reaed

In [5]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

In [6]:
image_size = 200

path1 = r"E:\Imarticus\images\facemaskface"
cate = ['face_detector' , 'facemask' ]

input_image=[] 
for i in cate:
    folders = os.path.join(path1,i)
    label   = cate.index(i)  # we need to tell software which image is of catand which is ofdog
    for image in os.listdir(folders):
        image_path = os.path.join(folders , image)
        image_array =cv2.imread(image_path) # using the cv2 i am reading the image and storing in variable image
        image_array = cv2.resize(image_array , (image_size, image_size))  # resizing each image to50*50
        input_image.append([image_array, label])
        

In [7]:
np.random.shuffle(input_image)
X = []
Y = []
for X_values , labels in input_image:
    X.append(X_values)
    Y.append(labels)

# seprate pixcells (images) and Y (0,1)`

In [8]:
len(X)

3833

In [9]:
3833 *.8

3066.4

In [10]:
x_train = X[0:3066]
y_train = Y[0:3066]

x_test = X[3066::]
y_test = Y[3066::]

In [11]:
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)

In [12]:
x_train= x_train/255 # this is to normalize the data 
x_test= x_test/255

In [15]:
base_model = VGG16(include_top=False , weights='imagenet',input_shape=(image_size,image_size,3))

model = Sequential()

model.add(base_model)

model.add(Flatten())
model.add(Dense(128 , activation= 'relu' ))
model.add(Dropout(0.4))
model.add(Dense(2 , activation= 'softmax'))

In [16]:
initial_lr = .001
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay( initial_learning_rate = initial_lr ,decay_steps = 1000,decay_rate= .9)

In [2]:
77*32*.8

1971.2

In [18]:

model.compile( optimizer= 'adam' , loss = 'sparse_categorical_crossentropy' , metrics = ['accuracy'])
model.fit(x_train , y_train , epochs = 10 , validation_split=.2 , batch_size= 32)

Epoch 1/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 2898s 38s/step - accuracy: 0.7251 - loss: 0.8341 - val_accuracy: 0.7427 - val_loss: 0.6025
Epoch 2/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 835s 10s/step - accuracy: 0.8927 - loss: 0.2862 - val_accuracy: 0.9202 - val_loss: 0.2298
Epoch 3/10
77/77 ━━━━━━━━━━━━━━━━━━━━ 810s 10s/step - accuracy: 0.9164 - loss: 0.2377 - val_accuracy: 0.8860 - val_loss: 0.2613
Epoch 4/10
20/77 ━━━━━━━━━━━━━━━━━━━━ 9:09 10s/step - accuracy: 0.8889 - loss: 0.3085

KeyboardInterrupt: 

In [19]:
pred = model.predict(x_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step


In [20]:
pred_classes = pred.argmax(axis = 1 ) # argmax  on each row

In [21]:
from  sklearn.metrics import confusion_matrix , classification_report
print(confusion_matrix(y_test, pred_classes))
print(classification_report(y_test , pred_classes))

[[334  47]
 [ 19 367]]
              precision    recall  f1-score   support

           0       0.95      0.88      0.91       381
           1       0.89      0.95      0.92       386

    accuracy                           0.91       767
   macro avg       0.92      0.91      0.91       767
weighted avg       0.92      0.91      0.91       767



In [22]:
model.save("facemask_VGG16.h5")